In [ ]:
from __future__ import annotations

import gc
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

try:
    import torch
except Exception:  # pragma: no cover
    torch = None


OUTER_FOLDS = (1, 2, 3, 4, 5)
SEED = 42
TARGET_COL = "Target_Log"
MODEL_ID = "tabpfn_v26"
RUN_ROLE = "default"
EXPLAIN_N_REPEATS = 5
EXPLAIN_SAMPLE_N = 5000


In [ ]:
def resolve_code_dir() -> Path:
    if "__file__" in globals():
        return Path(__file__).resolve().parent
    code_dir = Path.cwd().resolve()
    if code_dir.name != "1_Code":
        raise RuntimeError("current working directory must be Step6_TabPFNExplainability/1_Code")
    return code_dir


CODE_DIR = resolve_code_dir()
STEP_ROOT = CODE_DIR.parent
RELEASE_ROOT = STEP_ROOT.parent
RESULT_DIR = STEP_ROOT / "2_Results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

PER_FOLD_PATH = RESULT_DIR / "0_tabpfn_winner_full_feature_importance_per_fold.csv"
SUMMARY_PATH = RESULT_DIR / "0_tabpfn_winner_full_feature_borda_ranking.csv"
RAW_SSP_PATH = RESULT_DIR / "0_tabpfn_winner_raw_ssp_borda_ranking.csv"

if not (RELEASE_ROOT / "Step5_ModelComparison").is_dir():
    raise FileNotFoundError(f"cannot resolve clean release root from {CODE_DIR}")

print("Step5.5 paths ready")
print("RESULT_DIR=", RESULT_DIR)


In [ ]:
from tabpfn.constants import ModelVersion

SUPPORTED_MODEL_VERSIONS = [name for name in dir(ModelVersion) if name.startswith("V")]
if "V2_6" not in SUPPORTED_MODEL_VERSIONS:
    raise RuntimeError("tabpfn_env must expose ModelVersion.V2_6; fallback is not allowed")
if torch is not None:
    torch.set_num_threads(max(1, min(8, torch.get_num_threads())))

print("TabPFN runtime ready:", SUPPORTED_MODEL_VERSIONS)


In [ ]:
def require_file(path: Path) -> Path:
    if not path.is_file():
        raise FileNotFoundError(str(path))
    return path


def load_feature_columns(path: Path) -> list[str]:
    df = pd.read_csv(require_file(path))
    if "feature" not in df.columns:
        raise KeyError(f"feature column missing from {path}")
    features = df["feature"].astype(str).tolist()
    if len(features) != 33:
        raise RuntimeError(f"full-feature contract must contain 33 features, got {len(features)}")
    return features


def load_full_feature_test(outer_fold: int) -> tuple[pd.DataFrame, list[str]]:
    processed_dir = RELEASE_ROOT / "Step4_ModelInputs" / "2_Results" / f"fold_{outer_fold}" / "full_feature_unified" / "processed"
    test_df = pd.read_csv(require_file(processed_dir / "0_test_processed.csv"))
    features = load_feature_columns(processed_dir / "0_feature_columns.csv")
    required = ["row_uid", "Country Code", "Year", "WasteFlag", TARGET_COL, *features]
    missing = [col for col in required if col not in test_df.columns]
    if missing:
        raise KeyError(f"fold {outer_fold} test input missing columns: {missing}")
    return test_df, features


print("Input loader functions ready")


In [ ]:
def load_artifact_manifest() -> pd.DataFrame:
    manifest_path = RELEASE_ROOT / "Step5_ModelComparison" / "2_Results" / "3_tabpfn_family_compare" / "3_tabpfn_model_artifact_manifest.csv"
    manifest = pd.read_csv(require_file(manifest_path))
    expected = {"outer_fold", "model_id", "run_role", "artifact_path"}
    missing = sorted(expected.difference(manifest.columns))
    if missing:
        raise KeyError(f"artifact manifest missing columns: {missing}")
    manifest = manifest[(manifest["model_id"].astype(str) == MODEL_ID) & (manifest["run_role"].astype(str) == RUN_ROLE)].copy()
    if set(manifest["outer_fold"].astype(int)) != set(OUTER_FOLDS):
        raise RuntimeError("artifact manifest must contain exactly folds 1..5 for tabpfn_v26 default")
    manifest["artifact_path"] = manifest["artifact_path"].astype(str)
    for artifact in manifest["artifact_path"]:
        require_file(Path(artifact))
    return manifest.sort_values("outer_fold").reset_index(drop=True)


ARTIFACT_MANIFEST = load_artifact_manifest()
print("Artifact manifest ready")
print(ARTIFACT_MANIFEST.to_string(index=False))


In [ ]:
def load_saved_model(outer_fold: int):
    row = ARTIFACT_MANIFEST[ARTIFACT_MANIFEST["outer_fold"].astype(int) == int(outer_fold)]
    if len(row) != 1:
        raise RuntimeError(f"expected one model artifact for fold {outer_fold}")
    artifact_path = Path(row.iloc[0]["artifact_path"])
    model = joblib.load(artifact_path)
    if not hasattr(model, "predict"):
        raise TypeError(f"loaded artifact has no predict method: {artifact_path}")
    return model, artifact_path


def prepare_eval_frame(test_df: pd.DataFrame, features: list[str], outer_fold: int) -> tuple[pd.DataFrame, pd.Series]:
    eval_df = test_df.drop_duplicates(subset=["row_uid"], keep="first").reset_index(drop=True)
    sample_n = min(EXPLAIN_SAMPLE_N, len(eval_df))
    if sample_n < len(eval_df):
        eval_df = eval_df.sample(n=sample_n, random_state=SEED + int(outer_fold)).reset_index(drop=True)
    x_eval = eval_df.loc[:, features].apply(pd.to_numeric, errors="coerce")
    x_eval = x_eval.fillna(x_eval.median(axis=0).fillna(0.0))
    y_eval = pd.to_numeric(eval_df[TARGET_COL], errors="raise").astype(float)
    if not np.isfinite(x_eval.to_numpy(dtype=float)).all():
        raise ValueError(f"fold {outer_fold} evaluation features contain non-finite values")
    if not np.isfinite(y_eval.to_numpy(dtype=float)).all():
        raise ValueError(f"fold {outer_fold} evaluation target contains non-finite values")
    return x_eval, y_eval


def clear_runtime_memory() -> None:
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


print("Model and eval helpers ready")


In [ ]:
def run_one_fold_importance(outer_fold: int) -> pd.DataFrame:
    test_df, features = load_full_feature_test(outer_fold)
    model, artifact_path = load_saved_model(outer_fold)
    x_eval, y_eval = prepare_eval_frame(test_df, features, outer_fold)
    print(f"Fold {outer_fold}: start permutation importance; rows={len(x_eval)}, features={len(features)}, repeats={EXPLAIN_N_REPEATS}", flush=True)
    result = permutation_importance(
        model,
        x_eval,
        y_eval,
        n_repeats=EXPLAIN_N_REPEATS,
        random_state=SEED,
        scoring="neg_mean_absolute_error",
    )
    fold_df = pd.DataFrame(
        {
            "outer_fold": int(outer_fold),
            "artifact_path": str(artifact_path),
            "feature": features,
            "importance_mean": result.importances_mean.astype(float),
            "importance_std": result.importances_std.astype(float),
        }
    )
    fold_df = fold_df.sort_values(["importance_mean", "feature"], ascending=[False, True], kind="mergesort").reset_index(drop=True)
    fold_df["rank_in_fold"] = np.arange(1, len(fold_df) + 1, dtype=int)
    fold_df["borda_score"] = len(fold_df) - fold_df["rank_in_fold"] + 1
    fold_path = RESULT_DIR / f"0_fold_{outer_fold}_tabpfn_winner_importance.csv"
    fold_df.to_csv(fold_path, index=False)
    clear_runtime_memory()
    print(f"Fold {outer_fold}: saved {fold_path}", flush=True)
    print(fold_df.head(10).to_string(index=False))
    return fold_df


print("Fold runner ready")


In [ ]:
fold_1_importance = run_one_fold_importance(1)


In [ ]:
fold_2_importance = run_one_fold_importance(2)


In [ ]:
fold_3_importance = run_one_fold_importance(3)


In [ ]:
fold_4_importance = run_one_fold_importance(4)


In [ ]:
fold_5_importance = run_one_fold_importance(5)


In [ ]:
def load_completed_fold_tables() -> pd.DataFrame:
    paths = [RESULT_DIR / f"0_fold_{outer_fold}_tabpfn_winner_importance.csv" for outer_fold in OUTER_FOLDS]
    missing = [str(path) for path in paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"missing completed fold importance files: {missing}")
    per_fold = pd.concat([pd.read_csv(path) for path in paths], ignore_index=True)
    if set(per_fold["outer_fold"].astype(int)) != set(OUTER_FOLDS):
        raise RuntimeError("per-fold importance does not cover folds 1..5")
    feature_counts = per_fold.groupby("outer_fold")["feature"].nunique().to_dict()
    if set(feature_counts.values()) != {33}:
        raise RuntimeError(f"each fold must contain 33 features, got {feature_counts}")
    return per_fold


per_fold_importance = load_completed_fold_tables()
per_fold_importance.to_csv(PER_FOLD_PATH, index=False)
print("Merged per-fold table saved:", PER_FOLD_PATH)


In [ ]:
def build_borda_summary(per_fold: pd.DataFrame) -> pd.DataFrame:
    summary = (
        per_fold.groupby("feature", as_index=False)
        .agg(
            borda_score_sum=("borda_score", "sum"),
            importance_mean_5fold_avg=("importance_mean", "mean"),
            importance_std_5fold_avg=("importance_std", "mean"),
            rank_mean_5fold=("rank_in_fold", "mean"),
        )
        .sort_values(["borda_score_sum", "importance_mean_5fold_avg", "feature"], ascending=[False, False, True], kind="mergesort")
        .reset_index(drop=True)
    )
    summary.insert(0, "borda_rank", np.arange(1, len(summary) + 1, dtype=int))
    if len(summary) != 33:
        raise RuntimeError(f"summary must contain 33 features, got {len(summary)}")
    return summary


borda_summary = build_borda_summary(per_fold_importance)
borda_summary.to_csv(SUMMARY_PATH, index=False)
print("Borda summary saved:", SUMMARY_PATH)
borda_summary.head(20)


In [ ]:
RAW_SSP_FEATURES = {
    "EG.ELC.ACCS.ZS",
    "EN.ATM.PM25.MC.M3",
    "EN.GHG.CO2.MT.CE.AR5",
    "EN.POP.DNST",
    "NV.IND.TOTL.ZS",
    "NV.SRV.TOTL.ZS",
    "NY.GDP.MKTP.KD.ZG",
    "SP.DYN.LE00.IN",
    "SP.POP.GROW",
    "SP.POP.TOTL",
    "SP.URB.GROW",
    "SP.URB.TOTL.IN.ZS",
    "NY.GDP.MKTP.PP.KD",
    "NY.GDP.PCAP.PP.KD",
}

raw_ssp_summary = borda_summary[borda_summary["feature"].isin(RAW_SSP_FEATURES)].copy().reset_index(drop=True)
raw_ssp_summary.insert(0, "raw_ssp_rank", np.arange(1, len(raw_ssp_summary) + 1, dtype=int))
if raw_ssp_summary.empty:
    raise RuntimeError("raw SSP Borda summary must not be empty")
raw_ssp_summary.to_csv(RAW_SSP_PATH, index=False)
print("Raw SSP summary saved:", RAW_SSP_PATH)
raw_ssp_summary
